In [ ]:
import numpy as np

# =============================================================================
# Berry Phase Truncation Error Bound
#
# Bound: |Δγ_N| ≤ Σ_j arctan( ε_j·ε_{j+1} / ((1-ε_j)(1-ε_{j+1})|O_D(j)| )
#
# Phase model from Fig. 4B (slope ±1/2 reading):
#   arg(β/α) = (θ - π)/2  →  a_11(θ) = √p_11 · e^{i(θ-π)/2}
#   arg(γ/α) = (π - θ)/2  →  a_20(θ) = √p_20 · e^{i(π-θ)/2}
#   α taken as real gauge:  a_02(θ) = √p_02
# =============================================================================

# -----------------------------------------------------------------------------
# Probability vectors from hardware (Hanoi, Apr 4)
# Format: (angle θ, probability vector)
# Basis ordering: |00>,|01>,|02>,|10>,|11>,|12>,|20>,|21>,|22>
# Dominant indices: 2 (|02>), 4 (|11>), 6 (|20>)
# -----------------------------------------------------------------------------

w = {}
w[0]  = (0,              np.array([ 0.01162158, -0.00465488,  0.33739193,  0.0072317,
                                     0.31953348,  0.00605047,  0.30543009,  0.02578479, -0.00838916]))
w[1]  = (2*np.pi/3,      np.array([ 0.00067024, -0.00236954,  0.43210499, -0.00177051,
                                     0.10078978,  0.01457336,  0.39321297,  0.03774422,  0.02504449]))
w[2]  = (np.pi/2,        np.array([ 2.12115478e-02,  3.46200781e-03,  3.47218970e-01,  1.57590447e-02,
                                     1.84381092e-01,  1.61194434e-04,  3.57202571e-01,  2.98836136e-02,  4.07199590e-02]))
w[3]  = (2*np.pi/5,      np.array([ 0.03142523,  0.00194623,  0.35768344,  0.00328126,
                                     0.21747801,  0.01430082,  0.32693466,  0.02654251,  0.02040784]))
w[4]  = (np.pi/3,        np.array([ 2.21261481e-02, -1.36740039e-04,  3.20134642e-01,  4.79971741e-03,
                                     2.67555581e-01,  4.67825507e-03,  3.39997716e-01,  2.26352755e-02,  1.82094055e-02]))
w[5]  = (2*np.pi/7,      np.array([ 0.01091598, -0.00241662,  0.35073897,  0.01036843,
                                     0.29054482, -0.00758829,  0.29930806,  0.03246427,  0.01566437]))
w[6]  = (4*np.pi/3,      np.array([ 0.01443338, -0.00759534,  0.39960547,  0.0082209,
                                     0.09052253, -0.00192335,  0.39951553,  0.03615589,  0.06106499]))
w[7]  = (4*np.pi/5,      np.array([ 6.80718910e-03,  6.02283556e-03,  4.43831627e-01, -3.66688786e-03,
                                     6.38676080e-02,  2.86909548e-02,  4.20161469e-01,  3.43499846e-02, -6.47802471e-05]))
w[8]  = (4*np.pi/7,      np.array([ 0.02959617,  0.00259343,  0.36817979,  0.00788336,
                                     0.18920559,  0.00494269,  0.39323909,  0.02657886, -0.022219]))
w[9]  = (3*np.pi/2,      np.array([ 0.00722698,  0.00380351,  0.39609568, -0.00232187,
                                     0.19588625,  0.02059617,  0.34785905,  0.02012307,  0.01073116]))
w[10] = (6*np.pi/5,      np.array([ 0.01687536,  0.00418956,  0.4172253,  -0.00176515,
                                     0.04644531,  0.02833333,  0.40482875,  0.03107608,  0.05279148]))
w[11] = (6*np.pi/7,      np.array([ 0.02584046,  0.00547339,  0.41774764, -0.00507892,
                                     0.02374591,  0.0311637,   0.42527601,  0.02961148,  0.04622033]))
w[12] = (5*np.pi/3,      np.array([ 0.005196,    0.00157121,  0.37354749,  0.002142,
                                     0.2548247,   0.01558938,  0.31573322,  0.02752265,  0.00387336]))
w[13] = (8*np.pi/7,      np.array([ 0.02349984, -0.0057321,   0.43988562, -0.00135873,
                                     0.04005334,  0.0423582,   0.44576959,  0.03916698, -0.02364274]))
w[14] = (10*np.pi/7,     np.array([ 0.01175623,  0.00166792,  0.37777297,  0.0122775,
                                     0.16938342,  0.00785588,  0.39387426,  0.03347079, -0.00805898]))
w[15] = (12*np.pi/7,     np.array([ 0.01449329, -0.00143568,  0.34005056,  0.00144727,
                                     0.23476494,  0.01983087,  0.33024341,  0.03142149,  0.02918384]))
w[16] = (np.pi,          np.array([-0.01654823,  0.01369785,  0.45452254,  0.0087674,
                                    -0.00122485,  0.02452292,  0.47829132,  0.02643333,  0.01153772]))

# Dominant component indices: |02>=2, |11>=4, |20>=6
DOM = [2, 4, 6]

# -----------------------------------------------------------------------------
# Helper functions
# -----------------------------------------------------------------------------

def dominant_weight(prob):
    """Sum of dominant component probabilities."""
    return sum(max(prob[i], 0) for i in DOM)

def epsilon(prob):
    """Non-dominant population weight ε = 1 - Σ p_dominant."""
    return 1.0 - dominant_weight(prob)

def complex_overlap(theta_j, prob_j, theta_k, prob_k):
    """
    Complex overlap <d_j|d_k> between normalized dominant states.

    Phase model (Fig. 4B, slope ±1/2):
        a_02 = √p_02            (real, gauge choice)
        a_11 = √p_11 · e^{i(θ-π)/2}
        a_20 = √p_20 · e^{i(π-θ)/2}

    <d_j|d_k> = [A_02 + A_11·e^{iΔθ/2} + A_20·e^{-iΔθ/2}] / √(dom_j·dom_k)
    where Δθ = θ_k - θ_j
    """
    p02j = max(prob_j[2], 0); p11j = max(prob_j[4], 0); p20j = max(prob_j[6], 0)
    p02k = max(prob_k[2], 0); p11k = max(prob_k[4], 0); p20k = max(prob_k[6], 0)
    dj = dominant_weight(prob_j)
    dk = dominant_weight(prob_k)

    A02 = np.sqrt(p02j * p02k)
    A11 = np.sqrt(p11j * p11k)
    A20 = np.sqrt(p20j * p20k)
    dtheta = (theta_k - theta_j) / 2.0  # slope-1/2 per Fig. 4B

    z = A02 + A11 * np.exp(1j * dtheta) + A20 * np.exp(-1j * dtheta)
    return z / np.sqrt(dj * dk)

def per_step_bound(theta_j, prob_j, theta_k, prob_k):
    """Per-step truncation error bound at step (j, j+1)."""
    ov   = complex_overlap(theta_j, prob_j, theta_k, prob_k)
    OD   = abs(ov)**2          # |<d_j|d_{j+1}>|^2 = Tr(rho_D(j) rho_D(j+1))
    ej   = epsilon(prob_j)
    ek   = epsilon(prob_k)
    return np.arctan(ej * ek / ((1 - ej) * (1 - ek) * OD))

# -----------------------------------------------------------------------------
# Partition configurations (from notebook)
# Each entry: (w_index, angle_override or None)
# angle_override needed when same probability vector is reused at a different θ
# (exploiting symmetry p(θ) = p(2π-θ) for AKLT ground states)
# -----------------------------------------------------------------------------

configs = {
    'N=3': [(0, None), (1,  None), (6,  None), (0,  2*np.pi)],
    'N=4': [(0, None), (2,  None), (16, None), (9,  None),   (0,  2*np.pi)],
    'N=5': [(0, None), (3,  None), (7,  None), (10, None),   (3,  8*np.pi/5), (0, 2*np.pi)],
    'N=6': [(0, None), (4,  None), (1,  None), (16, None),   (6,  None), (12, None), (0, 2*np.pi)],
    'N=7': [(0, None), (5,  None), (8,  None), (11, None),   (13, None), (14, None), (15, None), (0, 2*np.pi)],
}

# -----------------------------------------------------------------------------
# Compute bounds
# -----------------------------------------------------------------------------

print("=" * 80)
print("BERRY PHASE TRUNCATION ERROR BOUND")
print("  |Δγ_N| ≤ Σ_j arctan( ε_j·ε_{j+1} / ((1-ε_j)(1-ε_{j+1})·|O_D(j)|) )")
print("=" * 80)

summary = {}

for N_label, seq in configs.items():
    print(f"\n{N_label}  (code N = {len(seq)}, overlaps = {len(seq)-1})")
    print(f"  {'Step':>4}  {'θ_j/π':>7}  {'θ_{j+1}/π':>9}  {'ε_j':>7}  {'ε_{j+1}':>8}  "
          f"{'|O_D|':>7}  {'per-step bound':>14}")
    print("  " + "-" * 70)

    total_bound = 0.0
    O_min       = np.inf
    step_data   = []

    for i in range(len(seq) - 1):
        idx_j, ang_j = seq[i]
        idx_k, ang_k = seq[i + 1]
        theta_j = ang_j if ang_j is not None else w[idx_j][0]
        theta_k = ang_k if ang_k is not None else w[idx_k][0]
        prob_j  = w[idx_j][1]
        prob_k  = w[idx_k][1]

        ov      = complex_overlap(theta_j, prob_j, theta_k, prob_k)
        OD      = abs(ov)**2
        ej      = epsilon(prob_j)
        ek      = epsilon(prob_k)
        ps      = per_step_bound(theta_j, prob_j, theta_k, prob_k)

        total_bound += ps
        O_min        = min(O_min, OD)
        step_data.append((theta_j, theta_k, ej, ek, OD, ps))

        print(f"  {i+1:>4}  {theta_j/np.pi:>7.4f}  {theta_k/np.pi:>9.4f}  "
              f"{ej:>7.4f}  {ek:>8.4f}  {OD:>7.4f}  {ps:>12.6f} rad")

    summary[N_label] = (total_bound, O_min)
    print(f"  {'':>4}  {'':>7}  {'':>9}  {'':>7}  {'':>8}  "
          f"O_min={O_min:.4f}  TOTAL = {total_bound:.4f} rad  ({100*total_bound/np.pi:.1f}% of π)")

# -----------------------------------------------------------------------------
# Summary table
# -----------------------------------------------------------------------------

print("\n" + "=" * 80)
print("SUMMARY")
print(f"  {'N':>4}  {'Bound (rad)':>12}  {'% of π':>8}")
print("  " + "-" * 28)
worst = 0.0
for N_label, (total, O_min) in summary.items():
    print(f"  {N_label:>4}  {total:>12.4f}  {100*total/np.pi:>7.1f}%")
    worst = max(worst, total)

print(f"\n  Worst-case bound across all N: {worst:.4f} rad ({100*worst/np.pi:.1f}% of π)")
print("=" * 80)


BERRY PHASE TRUNCATION ERROR BOUND
  |Δγ_N| ≤ Σ_j arctan( ε_j·ε_{j+1} / ((1-ε_j)(1-ε_{j+1})·|O_D(j)|) )

N=3  (code N = 4, overlaps = 3)
  Step    θ_j/π  θ_{j+1}/π      ε_j   ε_{j+1}    |O_D|  per-step bound
  ----------------------------------------------------------------------
     1   0.0000     0.6667   0.0376    0.0739   0.4900      0.006369 rad
     2   0.6667     1.3333   0.0739    0.1104   0.6134      0.016133 rad
     3   1.3333     2.0000   0.1104    0.0376   0.4872      0.009960 rad
                                               O_min=0.4872  TOTAL = 0.0325 rad  (1.0% of π)

N=4  (code N = 5, overlaps = 4)
  Step    θ_j/π  θ_{j+1}/π      ε_j   ε_{j+1}    |O_D|  per-step bound
  ----------------------------------------------------------------------
     1   0.0000     0.5000   0.0376    0.1112   0.6577      0.007441 rad
     2   0.5000     1.0000   0.1112    0.0672   0.6765      0.013319 rad
     3   1.0000     1.5000   0.0672    0.0602   0.6743      0.006837 rad
     4   1.